# Best place to be

## Bibliothèques

In [13]:
import pandas as pd
import zipfile
import numpy as np
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
import plotly.express as px
import hdbscan
from sklearn.preprocessing import MinMaxScaler

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)



## Chargement et préparation des données

In [14]:
# Chargement
zip_path = "D:/Profils/NLefort/Desktop/JEDHA/PROJETS/06.ML_non_supervise/uber-trip-data.zip"
sample_size = 10000
sample_df = pd.DataFrame()

with zipfile.ZipFile(zip_path) as z:
    for file_name in z.namelist(): # pour chaque fichier de la liste
        if "taxi-zone-lookup" in file_name.lower(): 
            continue # Ignore celui-là
        if file_name.endswith('.csv') and 'uber-raw-data' in file_name.lower(): # et si tu trouves un fichier .csv et qui s'apelle ...
            df = pd.read_csv(z.open(file_name), encoding='latin1') #lit-le
            
            n_sample = min(sample_size, len(df)) # depuis ces fichiers lus, conserve un échantillon
            sample_df = pd.concat([sample_df, df.sample(n=n_sample, random_state=42)], ignore_index=True)

print(sample_df.head())
print(f"Échantillon final de {len(sample_df)} lignes")


           Date/Time      Lat      Lon    Base Unnamed: 0
0  4/9/2014 10:21:00  40.8021 -73.9654  B02598        NaN
1  4/14/2014 4:55:00  40.6462 -73.7769  B02764        NaN
2  4/23/2014 9:52:00  40.7747 -73.9603  B02598        NaN
3  4/4/2014 23:32:00  40.7150 -74.0157  B02682        NaN
4  4/5/2014 19:57:00  40.7335 -74.0080  B02598        NaN
Échantillon final de 60000 lignes


In [15]:
# Describe
print(sample_df.describe())
print()

# Filtrer les coordonnées aberrantes (Lat=40.712784, Lon=-74.005941)
df = sample_df[
    (sample_df['Lat'].between(40.5, 41)) &
    (sample_df['Lon'].between(-74.3, -73.7))
].copy()

# Valeurs manquantes
missing_values = df[['Date/Time','Lat', 'Lon', 'Base']].isnull().sum()
print(missing_values)


                Lat           Lon
count  60000.000000  60000.000000
mean      40.739275    -73.973557
std        0.039618      0.056464
min       40.122200    -74.654200
25%       40.721100    -73.996700
50%       40.742500    -73.983500
75%       40.761100    -73.966200
max       41.147800    -72.700600

Date/Time    0
Lat          0
Lon          0
Base         0
dtype: int64


In [16]:
# Convertir Fate/Time
df['Date/Time'] = pd.to_datetime(df['Date/Time'])

# Ajout jour + heure au df
df['day'] = df['Date/Time'].dt.day_name()
df['hour'] = df['Date/Time'].dt.hour

print(f"Dataset filtré :{len(df)} points")

Dataset filtré :59683 points


## Modèles

In [18]:
# Sélection d'un batch pour le modèle (mercredi, 9h)
day="Wednesday"
hour = 9

df_batch = df[(df['day'] == day) & (df['hour'] == hour)].copy()
print(f"{day}, {hour} h -> {len(df_batch)} points")

coords = df_batch[['Lat', 'Lon']].values

Wednesday, 9 h -> 329 points


In [19]:
# MiniBatchKMeans (optimisation de k)
best_k, best_score, best_labels = None, -1, None

for k in range (2,11): # Pour chaque k (entre 1 et 10)
    mbk = MiniBatchKMeans(n_clusters=k, batch_size=5000, random_state=42) # J'applique modèle, 1 lot = 10 000 point à chaque itération pour mettre à jour le centre du cluster
    labels = mbk.fit_predict(coords) # étiquettes = coordonnées prédites par mon modèle
    score = silhouette_score(coords, labels) # score silhouette dépend des valeurs prédites coordonnées vs étiquettes prédites
    print(f"k={k}, silhouette={score:.3f}")
    if score > best_score:
        best_k, best_score, best_labels = k, score, labels

df_batch['mbk_cluster']=best_labels
print(f"Meilleur k={best_k} avec silhouette={best_score:.3f}")

c:\Users\NLefort\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1955: UserWarning: MiniBatchKMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can prevent it by setting batch_size >= 3072 or by setting the environment variable OMP_NUM_THREADS=2
  warnings.warn(
c:\Users\NLefort\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1955: UserWarning: MiniBatchKMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can prevent it by setting batch_size >= 3072 or by setting the environment variable OMP_NUM_THREADS=2
  warnings.warn(
c:\Users\NLefort\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1955: UserWarning: MiniBatchKMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can prevent it by setting batch_size >= 3072 or by setting the environment var

k=2, silhouette=0.610
k=3, silhouette=0.447
k=4, silhouette=0.450
k=5, silhouette=0.367
k=6, silhouette=0.404
k=7, silhouette=0.355


c:\Users\NLefort\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1955: UserWarning: MiniBatchKMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can prevent it by setting batch_size >= 3072 or by setting the environment variable OMP_NUM_THREADS=2
  warnings.warn(
c:\Users\NLefort\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1955: UserWarning: MiniBatchKMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can prevent it by setting batch_size >= 3072 or by setting the environment variable OMP_NUM_THREADS=2
  warnings.warn(
c:\Users\NLefort\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1955: UserWarning: MiniBatchKMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can prevent it by setting batch_size >= 3072 or by setting the environment var

k=8, silhouette=0.385
k=9, silhouette=0.354
k=10, silhouette=0.375
Meilleur k=2 avec silhouette=0.610


c:\Users\NLefort\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1955: UserWarning: MiniBatchKMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can prevent it by setting batch_size >= 3072 or by setting the environment variable OMP_NUM_THREADS=2
  warnings.warn(
c:\Users\NLefort\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1955: UserWarning: MiniBatchKMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can prevent it by setting batch_size >= 3072 or by setting the environment variable OMP_NUM_THREADS=2
  warnings.warn(


In [21]:
# HDBSCAN
scaler=MinMaxScaler() 
coords_scaled = scaler.fit_transform(coords) # J'appliquer un scaler aux coordoonnées

clusterer = hdbscan.HDBSCAN(min_cluster_size=10) # définition des cluster par mon modèle (au mini 30 observations)
hdb_labels = clusterer.fit_predict(coords_scaled)

df_batch['hdb_cluster'] = hdb_labels 

# Calcul du score silhouette (hors bruit = -1)
mask= hdb_labels!=-1
if mask.sum() > 1 and len(set(hdb_labels[mask])) > 1:
    sil_hdb = silhouette_score(coords_scaled[mask], hdb_labels[mask])
else:
    sil_hdb = None
print(f"HDBSCAN Silhouette ={sil_hdb}")

HDBSCAN Silhouette =0.43331882435026714


In [22]:
# Visualisation des clusters
fig1 = px.scatter_map(
    df_batch, lat="Lat", lon="Lon",
    color="mbk_cluster", zoom=10, map_style="carto-positron",
    title=f"MiniBatchKMeans (k={best_k}, {day} {hour}h)"
)
fig1.show()

fig2 = px.scatter_map(
    df_batch, lat="Lat", lon="Lon",
    color="hdb_cluster", zoom=10, map_style="carto-positron", 
    title=f"HDBSCAN (min_cluster_size=30, {day} {hour}h)"
)
fig2.show()

In [12]:
# Livrable final : Carte interactive avec Plotly montrant les zones chaudes. // Graphiques montrant la variation des zones par jour et heure.
# Analyse comparative : KMeans vs DBSCAN, forces et limites.
# Conseils pratiques : Commence avec un petit échantillon (ex. 1 jour) puis étends à toute la semaine. Ajuste k pour KMeans et eps pour DBSCAN pour optimiser les clusters. Heatmap + clusters = visualisation très parlante pour Uber.

## Modèle complet

In [23]:
# Fonction clustering
def cluster_batch(df_batch, day, hour):
    coords = df_batch[['Lat','Lon']].values

    # MiniBatchKMeans (optimisation de k)
    best_k, best_score, best_labels = None, -1, None
    for k in range(2, 11):
        mbk = MiniBatchKMeans(n_clusters=k, batch_size=5000, random_state=42)
        labels = mbk.fit_predict(coords)
        score = silhouette_score(coords, labels)
        if score > best_score:
            best_k, best_score, best_labels = k, score, labels
    df_batch['mbk_cluster'] = best_labels

    # HDBSCAN
    scaler = MinMaxScaler()
    coords_scaled = scaler.fit_transform(coords)
    clusterer = hdbscan.HDBSCAN(min_cluster_size=30)
    hdb_labels = clusterer.fit_predict(coords_scaled)
    df_batch['hdb_cluster'] = hdb_labels

    # silhouette HDBSCAN (sans bruit)
    mask = hdb_labels != -1
    if mask.sum() > 1 and len(set(hdb_labels[mask])) > 1:
        sil_hdb = silhouette_score(coords_scaled[mask], hdb_labels[mask])
    else:
        sil_hdb = None

    return {
        "day": day,
        "hour": hour,
        "n_points": len(df_batch),
        "mbk_best_k": best_k,
        "mbk_silhouette": best_score,
        "hdb_clusters": len(set(hdb_labels)) - (1 if -1 in hdb_labels else 0),
        "hdb_silhouette": sil_hdb
    }, df_batch

# ================================
# Boucle par jour et heure
# ================================
results = []
all_batches = []

for day in df['day'].unique():
    for hour in range(24):
        df_batch = df[(df['day'] == day) & (df['hour'] == hour)].copy()
        if len(df_batch) < 200:  # trop peu de données
            continue
        res, df_clustered = cluster_batch(df_batch, day, hour)
        results.append(res)
        all_batches.append(df_clustered)

df_results = pd.DataFrame(results)
print(df_results.head())

# ================================
# Visualisations globales
# ================================

# Variation des clusters KMeans
fig_kmeans = px.line(
    df_results, x="hour", y="mbk_best_k", color="day",
    title="Variation du nombre de clusters (MiniBatchKMeans)"
)
fig_kmeans.show()

# Variation des clusters HDBSCAN
fig_hdb = px.line(
    df_results, x="hour", y="hdb_clusters", color="day",
    title="Variation du nombre de clusters (HDBSCAN)"
)
fig_hdb.show()



c:\Users\NLefort\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1955: UserWarning:

MiniBatchKMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can prevent it by setting batch_size >= 3072 or by setting the environment variable OMP_NUM_THREADS=2

c:\Users\NLefort\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1955: UserWarning:

MiniBatchKMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can prevent it by setting batch_size >= 3072 or by setting the environment variable OMP_NUM_THREADS=2

c:\Users\NLefort\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1955: UserWarning:

MiniBatchKMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can prevent it by setting batch_size >= 3072 or by setting the environment variable OMP_NUM_THREADS=2

c:\U

         day  hour  n_points  mbk_best_k  mbk_silhouette  hdb_clusters  \
0  Wednesday     6       353           2        0.619574             2   
1  Wednesday     7       496           2        0.468906             2   
2  Wednesday     8       425           4        0.467782             0   
3  Wednesday     9       329           2        0.609500             0   
4  Wednesday    10       359           7        0.450905             2   

   hdb_silhouette  
0        0.707946  
1        0.302178  
2             NaN  
3             NaN  
4        0.639099  


* MiniBatchKMeans – k optimal par jour
1 ligne = représente le nombre optimal de clusters k pour le jour donné. Ce nombre est déterminé en maximisant le silhouette score pour ce jour.
- Variabilité selon le jour : Certains jours, k optimal est plus élevé (plus de zones chaudes distinctes). 
Par exemple, le mercredi à 9h ou le dimanche à 8h, il semble qu'il y ait plus de trafic.

- Si k reste relativement stable (ex. 5–6 clusters tous les jours), cela indique que la structure de la demande est assez similaire chaque jour.
- Si k varie beaucoup, ça montre une fluctuation de la demande selon le jour.

Interprétation pratique :
- Uber pourrait ajuster le nombre de chauffeurs à positionner dans chaque zone selon le jour.
- Les jours avec plus de clusters nécessitent potentiellement plus de chauffeurs répartis.

* HDBSCAN – nombre de clusters par jour
Chaque ligne indique le nombre réel de clusters détectés automatiquement par HDBSCAN, après avoir filtré le bruit (-1). HDBSCAN dépend de min_cluster_size et min_samples. Les jours avec beaucoup de clusters détectés peuvent avoir une demande plus dispersée.

Clusters vs bruit :
- HDBSCAN peut marquer certains points comme bruit (-1) → pas comptés dans le nombre de clusters.
- Un nombre de clusters faible avec beaucoup de bruit → forte concentration dans quelques zones, mais certains trajets isolés.

* Comparaison KMeans vs HDBSCAN :
- Means force k clusters → toujours k.
- HDBSCAN détecte le nombre naturel de clusters → plus flexible, mais peut générer du bruit.

Si HDBSCAN détecte moins de clusters que k KMeans, cela indique que certains clusters KMeans sont très proches ou peu denses.

In [24]:
# Exemple de carte : lundi 8h
example = all_batches[0]
fig_map = px.scatter_map(
    example, lat="Lat", lon="Lon",
    color="mbk_cluster", zoom=10, map_style="carto-positron",
    title=f"Zones chaudes - {example['day'].iloc[1]} {example['hour'].iloc[9]}h"
)
fig_map.show()

# ================================
# Analyse comparative
# ================================
print("\n=== Analyse comparative ===")
print("MiniBatchKMeans : rapide, scalable, nécessite de choisir k (silhouette score aide).")
print("HDBSCAN : détecte le nombre de clusters, gère le bruit, mais plus lent et sensible aux paramètres.")

print("\n=== Conseils pratiques ===")
print("- Commencer avec 1 jour/1h (modèle simple).")
print("- Étendre ensuite à toute la semaine (modèle complet).")
print("- Optimiser k pour KMeans et min_cluster_size pour HDBSCAN.")
print("- Utiliser les cartes Plotly pour interpréter les zones chaudes.")


=== Analyse comparative ===
MiniBatchKMeans : rapide, scalable, nécessite de choisir k (silhouette score aide).
HDBSCAN : détecte le nombre de clusters, gère le bruit, mais plus lent et sensible aux paramètres.

=== Conseils pratiques ===
- Commencer avec 1 jour/1h (modèle simple).
- Étendre ensuite à toute la semaine (modèle complet).
- Optimiser k pour KMeans et min_cluster_size pour HDBSCAN.
- Utiliser les cartes Plotly pour interpréter les zones chaudes.


## Optimisation

Étapes d’optimisation
1. Optimiser MiniBatchKMeans
* Actuellement, tests k de 2 à 10. Selon l’échelle, il peut y avoir plus de zones chaudes. Étendre la recherche de k (ex. 2 → 30).
* Utiliser silhouette score ou Davies-Bouldin index pour évaluer.
* Choisir un compromis entre un score correct et une interprétation lisible sur carte (pas trop de clusters).

2. Optimiser HDBSCAN
* HDBSCAN dépend surtout de min_cluster_size et min_samples.
* min_cluster_size : taille minimale d’un cluster (ex. 20 → 100).
* min_samples : robustesse face au bruit (plus grand = plus de points classés bruit).
* Astuce : tu peux boucler sur plusieurs valeurs et comparer le nombre de clusters + silhouette (hors bruit).

3. Optimiser par heure/jour
* Les patterns de clusters varient beaucoup selon l’heure (rush du matin ≠ soir ≠ nuit).

Donc :
- Calcule le meilleur k par heure/jour (MiniBatchKMeans).
- Compare avec le meilleur min_cluster_size par heure/jour (HDBSCAN).

Cela donnera un tableau comparatif des paramètres optimaux.

4. Gestion des 120k points

MiniBatchKMeans → scalable

HDBSCAN → peut être lent → tu peux :

travailler par batch (1 jour ou quelques heures).

réduire légèrement avec un sample mais en gardant la représentativité.

In [25]:
# Sélection : ex. lundi
day = "Monday"
df_day = df[df['day'] == day].copy()

print(f"{len(df_day)} points pour {day}")

# Fonction optimisation MiniBatchKMeans
def best_kmeans(coords, k_min=2, k_max=20):
    best_k, best_score, best_labels = None, -1, None
    for k in range(k_min, k_max+1):
        mbk = MiniBatchKMeans(n_clusters=k, batch_size=5000, random_state=42)
        labels = mbk.fit_predict(coords)
        sil = silhouette_score(coords, labels)
        db = davies_bouldin_score(coords, labels)
        score = sil - db  # compromis silhouette - DB
        if score > best_score:
            best_k, best_score, best_labels = k, score, labels
    return best_k, best_labels

# Fonction optimisation HDBSCAN
def best_hdbscan(coords_scaled, min_sizes=[20,50,100]):
    best_model, best_score, best_labels = None, -1, None
    for m in min_sizes:
        clusterer = hdbscan.HDBSCAN(min_cluster_size=m)
        labels = clusterer.fit_predict(coords_scaled)
        mask = labels != -1
        if mask.sum() > 1 and len(set(labels[mask])) > 1:
            sil = silhouette_score(coords_scaled[mask], labels[mask])
            if sil > best_score:
                best_model, best_score, best_labels = clusterer, sil, labels
    return best_model, best_labels

# Application sur le jour choisi
coords = df_day[['Lat','Lon']].values
scaler = MinMaxScaler()
coords_scaled = scaler.fit_transform(coords)

# MiniBatchKMeans optimisé
k_opt, labels_kmeans = best_kmeans(coords, k_min=2, k_max=20)
df_day['mbk_cluster'] = labels_kmeans
print(f"KMeans optimal : k={k_opt}")

# HDBSCAN optimisé
model_hdb, labels_hdb = best_hdbscan(coords_scaled, min_sizes=[20,50,100])
df_day['hdb_cluster'] = labels_hdb
print(f"HDBSCAN clusters détectés : {len(set(labels_hdb)) - (1 if -1 in labels_hdb else 0)}")

# Carte comparative
fig1 = px.scatter_map(
    df_day, lat="Lat", lon="Lon",
    color="mbk_cluster", zoom=10, map_style="carto-positron",
    title=f"Zones chaudes - {day} (MiniBatchKMeans k={k_opt})"
)
fig1.show()

fig2 = px.scatter_map(
    df_day, lat="Lat", lon="Lon",
    color="hdb_cluster", zoom=10, map_style="carto-positron",
    title=f"Zones chaudes - {day} (HDBSCAN)"
)
fig2.show()

6967 points pour Monday
KMeans optimal : k=4
HDBSCAN clusters détectés : 13


In [36]:
df.shape

(59683, 7)

In [35]:
results = []

for day in df['day'].unique():
    df_day = df[df['day'] == day].copy()
    coords = df_day[['Lat','Lon']].values
    scaler = MinMaxScaler()
    coord_scaled = scaler.fit_transform(coords)

    # KMeans optimisé
    k_opt, labels_kmeans = best_kmeans(coords, k_min=2, k_max=20)
    df_day['mbk_cluster'] = labels_kmeans

    # HDBSCAN optimisé
    model_hdb, labels_hdb = best_hdbscan(coord_scaled, min_sizes=[20,50,100])
    df_day['hdb_cluster'] = labels_hdb

    results.append({
        "day": day,
        "kmeans_k": k_opt,
        "hdb_clusters": len(set(labels_hdb)) - (1 if -1 in labels_hdb else 0)
    })

df_results = pd.DataFrame(results)
print(df_results)


KeyboardInterrupt: 

## Graphiques

In [ ]:
# Variation de k optimal
fig_kmeans = px.bar(
    df_results, x="day", y="kmeans_k",
    title="Variation de k optimal (MiniBatchKMeans) par jour",
    text="kmeans_k", color="kmeans_k"
)
fig_kmeans.update_traces(textposition="outside")
fig_kmeans.show()

In [ ]:
# Variation du nombre de clusters (HDBSCAN)
fig_hdb = px.bar(
    df_results, x="day", y="hdb_clusters",
    title="Variation du nombre de clusters (HDBSCAN) par jour",
    text="hdb_clusters", color="hdb_clusters"
)
fig_hdb.update_traces(textposition="outside")
fig_hdb.show()